# Prompt Sensitivity Study

**Research question:** Can better prompting close the modality gap between text-only and image-based reasoning?

## 8 Prompt Strategies

| # | Strategy | Category | Hypothesis |
|---|----------|----------|------------|
| 1 | Zero-shot | baseline | Original study's prompt |
| 2 | Zero-shot CoT | cot | "Let's think step by step" improves reasoning |
| 3 | Instruction priming | priming | "Read the image carefully" reduces OCR errors |
| 4 | Format priming | priming | "Extract numbers first" helps parsing |
| 5 | 1-shot | few_shot | One example teaches the answer format |
| 6 | 3-shot | few_shot | More examples = better calibration |
| 7 | Self-verify | verification | "Check your work" catches arithmetic errors |
| 8 | Structured output | structured | Forced GIVEN/FIND/STEPS template |

For each strategy, we run **both text-only and image conditions** and measure:
- Absolute accuracy improvement per strategy
- Whether the **modality gap** (text minus image) shrinks
- Which error types each strategy reduces

**Runtime:** ~20 min per strategy per condition (200 problems × 2 conditions × 8 strategies = ~5h)

In [ ]:
!pip install torch transformers datasets scipy pyyaml tqdm pillow bitsandbytes --quiet

In [ ]:
!git clone https://github.com/Ro-netizen004/vlm-modality-research.git /content/repo 2>/dev/null || echo 'exists'
import sys
sys.path.insert(0, '/content/repo')

from google.colab import drive
drive.mount('/content/drive')
OUTPUT_DIR = '/content/drive/MyDrive/vlm_research_results/prompt_sensitivity'
!mkdir -p {OUTPUT_DIR}

In [ ]:
# ══════════════════════════════════════════════════════════
#  CONFIGURATION
# ══════════════════════════════════════════════════════════

NUM_PROBLEMS = 200  # subset for tractability

MODEL = {
    'name': 'Qwen/Qwen2-VL-2B-Instruct',
    'type': 'qwen',
    'max_new_tokens': 512,  # higher for CoT/few-shot
    'torch_dtype': 'bfloat16',
    'quantize': None,
}

# Which strategies to run this session
STRATEGIES = [
    'zero_shot', 'zero_shot_cot', 'instruction_priming', 'format_priming',
    'few_shot_1', 'few_shot_3', 'self_verify', 'structured_output',
]

from src.prompts import list_strategies
list_strategies()

In [ ]:
import os, time, json, torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from datasets import load_dataset
from tqdm import tqdm
from PIL import Image

from src.models import VLMModel
from src.evaluation import (
    answers_match, classify_error, compute_accuracy,
    binomial_ci, bootstrap_ci, mcnemar_test, cohens_h,
)
from src.rendering import render_all_images, load_image
from src.prompts import get_prompts, PROMPT_STRATEGIES

torch.manual_seed(42)

# Load data
ds = load_dataset('gsm8k', 'main', split='test').select(range(NUM_PROBLEMS))
questions = list(ds['question'])
references = list(ds['answer'])
N = len(questions)

# Render images
IMAGE_DIR = os.path.join(OUTPUT_DIR, 'images')
render_all_images(questions, IMAGE_DIR)

print(f'Loaded {N} problems')

In [ ]:
# ══════════════════════════════════════════════════════════
#  RUN ALL PROMPT STRATEGIES
# ══════════════════════════════════════════════════════════

vlm = VLMModel(**MODEL)
vlm.load()
short = MODEL['name'].split('/')[-1]

all_results = {}

for strat_name in STRATEGIES:
    strat_info = PROMPT_STRATEGIES[strat_name]
    results_path = os.path.join(OUTPUT_DIR, f'{short}_{strat_name}.json')
    
    if os.path.exists(results_path):
        with open(results_path) as f:
            all_results[strat_name] = json.load(f)
        print(f'SKIP {strat_name} (cached): '
              f'text={all_results[strat_name]["acc_text"]:.3f} '
              f'image={all_results[strat_name]["acc_image"]:.3f}')
        continue
    
    print(f"\n{'='*60}")
    print(f'  Strategy: {strat_name} — {strat_info["description"]}')
    print(f"{'='*60}")
    
    text_correct, text_errors = [], []
    img_correct, img_errors = [], []
    
    # ── Text-only condition ──
    print(f'  Text-only:')
    for i, (q, ref) in enumerate(tqdm(zip(questions, references), total=N, desc='Text')):
        text_prompt, _ = get_prompts(strat_name, q)
        try:
            pred = vlm.generate_text_only(text_prompt if strat_name != 'zero_shot' else q)
            # For non-zero_shot strategies, we pass the full prompt directly
            # since generate_text_only wraps it differently per model
        except Exception as e:
            pred = f'ERROR: {e}'
        text_correct.append(answers_match(pred, ref))
        text_errors.append(classify_error(pred, ref))
    print(f'    Accuracy: {compute_accuracy(text_correct):.3f}')
    
    # ── Image condition ──
    print(f'  Image:')
    for i in tqdm(range(N), desc='Image'):
        _, image_prompt = get_prompts(strat_name, questions[i])
        try:
            img = load_image(i, IMAGE_DIR)
            pred = vlm.generate_with_image(img, text_prompt=image_prompt)
        except Exception as e:
            pred = f'ERROR: {e}'
        img_correct.append(answers_match(pred, references[i]))
        img_errors.append(classify_error(pred, references[i]))
    print(f'    Accuracy: {compute_accuracy(img_correct):.3f}')
    
    # ── Compute statistics ──
    acc_text = compute_accuracy(text_correct)
    acc_img = compute_accuracy(img_correct)
    gap = acc_text - acc_img  # positive = text better (vision hurts)
    mcn_chi2, mcn_p, mcn_b, mcn_c = mcnemar_test(text_correct, img_correct)
    effect = cohens_h(acc_text, acc_img)
    
    result = {
        'strategy': strat_name,
        'description': strat_info['description'],
        'category': strat_info['category'],
        'n': N,
        'acc_text': acc_text,
        'acc_image': acc_img,
        'modality_gap': gap,
        'text_ci_95': list(binomial_ci(sum(text_correct), N)),
        'img_ci_95': list(binomial_ci(sum(img_correct), N)),
        'mcnemar_p': mcn_p,
        'cohens_h': effect,
        'text_errors': dict(Counter(text_errors)),
        'img_errors': dict(Counter(img_errors)),
    }
    
    with open(results_path, 'w') as f:
        json.dump(result, f, indent=2)
    
    all_results[strat_name] = result
    print(f'  Gap: {gap*100:+.1f}pp  p={mcn_p:.4f}  h={effect:.3f}')

vlm.unload()
print('\nAll strategies complete!')

In [ ]:
# ══════════════════════════════════════════════════════════
#  VISUALIZATION: Modality Gap by Prompt Strategy
# ══════════════════════════════════════════════════════════

strats = list(all_results.keys())
text_accs = [all_results[s]['acc_text'] * 100 for s in strats]
img_accs = [all_results[s]['acc_image'] * 100 for s in strats]
gaps = [all_results[s]['modality_gap'] * 100 for s in strats]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: Text vs Image accuracy per strategy
ax = axes[0]
x = np.arange(len(strats))
w = 0.35
bars1 = ax.bar(x - w/2, text_accs, w, label='Text-Only', color='#4C72B0', edgecolor='black')
bars2 = ax.bar(x + w/2, img_accs, w, label='Image', color='#55A868', edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels([s.replace('_', '\n') for s in strats], fontsize=8, rotation=45, ha='right')
ax.set_ylabel('Accuracy (%)')
ax.set_title(f'Accuracy by Prompt Strategy — {short}')
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, max(max(text_accs), max(img_accs)) + 15)

# Panel 2: Modality gap (text - image)
ax = axes[1]
colors = ['#C44E52' if g > 5 else '#DDAA33' if g > 2 else '#55A868' for g in gaps]
bars = ax.barh(strats, gaps, color=colors, edgecolor='black')
for bar, g, p in zip(bars, gaps, [all_results[s]['mcnemar_p'] for s in strats]):
    sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'ns'
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{g:+.1f}pp {sig}', va='center', fontsize=9)
ax.set_xlabel('Modality Gap (pp) — Text minus Image')
ax.set_title('Does Prompting Close the Gap?')
ax.axvline(x=0, color='black', linewidth=0.5)
ax.grid(axis='x', alpha=0.3)

plt.suptitle('Prompt Sensitivity Study', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'{short}_prompt_sensitivity.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Error type shift: which strategies reduce arithmetic errors? ──

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

error_cats = ['correct', 'arithmetic_error', 'reasoning_error', 'no_number', 'vision_error']
colors = ['#55A868', '#4C72B0', '#8172B2', '#DDAA33', '#C44E52']

for ax_idx, (title, key) in enumerate([('Text-Only', 'text_errors'), ('Image', 'img_errors')]):
    ax = axes[ax_idx]
    bottoms = np.zeros(len(strats))
    for cat, color in zip(error_cats, colors):
        vals = [all_results[s][key].get(cat, 0) / all_results[s]['n'] * 100 for s in strats]
        ax.bar(strats, vals, bottom=bottoms, color=color, label=cat.replace('_', ' '),
               edgecolor='white', linewidth=0.5)
        bottoms += np.array(vals)
    ax.set_ylabel('Percentage (%)')
    ax.set_title(f'{title} Condition')
    if ax_idx == 0:
        ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
    ax.set_ylim(0, 105)
    ax.grid(axis='y', alpha=0.3)

axes[1].set_xticklabels([s.replace('_', '\n') for s in strats], fontsize=9, rotation=45, ha='right')
plt.suptitle(f'Error Distribution by Prompt Strategy — {short}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'{short}_error_by_strategy.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Summary table ──
rows = []
for s in strats:
    r = all_results[s]
    rows.append({
        'Strategy': s,
        'Category': r['category'],
        'Text Acc': f"{r['acc_text']:.3f}",
        'Image Acc': f"{r['acc_image']:.3f}",
        'Gap (pp)': f"{r['modality_gap']*100:+.1f}",
        'McNemar p': f"{r['mcnemar_p']:.4f}",
        "Cohen's h": f"{r['cohens_h']:.3f}",
        'Sig': '***' if r['mcnemar_p']<0.001 else '**' if r['mcnemar_p']<0.01 else '*' if r['mcnemar_p']<0.05 else 'ns',
    })

summary = pd.DataFrame(rows)
display(summary)
summary.to_csv(os.path.join(OUTPUT_DIR, f'{short}_prompt_summary.csv'), index=False)

# Key finding
baseline_gap = all_results['zero_shot']['modality_gap'] * 100
best_strat = min(all_results.keys(), key=lambda s: all_results[s]['modality_gap'])
best_gap = all_results[best_strat]['modality_gap'] * 100

print(f"\n{'='*60}")
print(f'Baseline modality gap: {baseline_gap:+.1f}pp')
print(f'Smallest gap: {best_strat} at {best_gap:+.1f}pp')
print(f'Gap reduction: {baseline_gap - best_gap:.1f}pp')
if best_gap > 5:
    print('Conclusion: Prompting CANNOT fully close the modality gap.')
elif best_gap > 2:
    print('Conclusion: Prompting partially closes the modality gap.')
else:
    print('Conclusion: Prompting effectively closes the modality gap!')
print(f"{'='*60}")